<a href="https://colab.research.google.com/github/RF-Lab/lab_sources/blob/master/DSP_S2_%D0%9F%D1%80%D0%B0%D0%BA%D1%82%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%BE%D0%B5_%D0%B7%D0%B0%D0%BD%D1%8F%D1%82%D0%B8%D0%B5_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
# см. Практическое занятие №7. Компенсация шума
# в сигнале ЭМГ на основе NLMS
#  Загрузка и анализ сигнала
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from tqdm import tqdm
pio.templates.default = 'presentation'
data = pd.read_csv(
'https://raw.githubusercontent.com/RF-Lab/'
'emg_platform/master/data/denoising/11111.txt',
index_col=False, sep=' ', header=None)
emgAndNoise = data.iloc[200:45000,0].to_numpy().flatten()
noiseOnly = data.iloc[200:45000,2].to_numpy().flatten()
noiseOnly-=np.mean(noiseOnly)
emgAndNoise-=np.mean(emgAndNoise)
fig = go.Figure()
fig.add_trace(go.Scatter(y=emgAndNoise,name='ЭМГ+Шум'))
fig.add_trace(go.Scatter(y=noiseOnly,name='Только шум'))
fig.show()

In [27]:
#  Подготовка матрицы X
M=40 # Длина фильтра
X = np.zeros((len(noiseOnly),M),dtype=np.float64)
for index,m in enumerate( range(M) ):
  X[:,index] = np.roll(noiseOnly,m)

In [31]:
# Алгоритм LMS из практического занятия 6 (Для сравнения)
N = len(noiseOnly)
h = np.zeros((M,N),dtype=np.float64)
yh = np.zeros_like(noiseOnly)
e = np.zeros_like(noiseOnly)
mu = 1e-10
for n in tqdm(range(N-1),desc='LMS',ncols=100):
  # Выход модели в момент времени n
  yh[n] = np.dot(X[n,:],h[:,n])
  # Ошибка
  e[n] = emgAndNoise[n] - yh[n]
  # шаг LMS
  h[:,n+1] = h[:,n] + mu*e[n]*X[n,:]
# Очищенный сигнал после алгоритма lms
lms_yh = e.copy()


# Алгоритм NLMS
h = np.zeros((M,N),dtype=np.float64)
e = np.zeros_like(noiseOnly)
mu = 8e-1
for n in tqdm(range(N-1),desc='NLMS',ncols=100):
  # Выход модели в момент времени n
  yh[n] = np.dot(X[n,:],h[:,n])
  # Ошибка
  e[n] = emgAndNoise[n] - yh[n]
  # шаг LMS
  h[:,n+1] = h[:,n] + mu/np.sum(np.square(X[n,:]))*e[n]*X[n,:]
# Очищенный сигнал после алгоритма lms
nlms_yh = e.copy()



fig = go.Figure()
fig.add_trace(go.Scatter(y=emgAndNoise,name='ЭМГ+Шум',line=dict(width=1)))
fig.add_trace(go.Scatter(y=lms_yh,name='После компенсации шума (LMS)',line=dict(width=1)))
fig.add_trace(go.Scatter(y=nlms_yh,name='После компенсации шума (NLMS)',line=dict(width=1,color='green')))


NLMS: 100%|████████████████████████████████████████████████| 44799/44799 [00:00<00:00, 73723.90it/s]
